<img src="imagens/banner.png" alt="Módulo 04: Seletores XPath e CSS" width="1100">

# Módulo 04: Seletores XPath e CSS

No Módulo 02, você viu que o navegador transforma o HTML em uma árvore, o DOM. No Módulo 03, aprendeu a baixar esse HTML com o requests, mas ele chegou como uma longa sequência de caracteres. Este módulo junta as duas coisas: a biblioteca **parsel** transforma o texto em árvore, e o **XPath** é a linguagem para caminhar por ela até o dado que você quer.

<img src="imagens/mapa-do-curso.png" alt="Mapa dos 10 módulos do curso, com o Módulo 04 destacado" width="1100">

## Neste módulo você vai

- Transformar o texto do HTML em uma árvore com o `Selector` do parsel;
- Escrever caminhos **XPath**, com condições (predicados) e funções;
- Escolher **o que extrair** de cada elemento: o texto, um atributo ou o elemento inteiro;
- Navegar pela árvore com os **eixos**: pai, irmão anterior, irmão seguinte, ancestrais;
- Partir do navegador (`F12`) para o código: inspecionar, entender a árvore e testar o XPath no console;
- Usar **contêineres** para extrair listas de itens, e tirar os dados dos **1000 livros** salvos no Módulo 03;
- Conhecer os **seletores CSS**, uma alternativa mais curta para os casos simples.

**Tempo estimado:** de 2h a 2h30.

## Como este módulo funciona

Como no Módulo 03, o módulo avança em passos pequenos: uma explicação curta e, logo abaixo, uma célula curta. Os exercícios **Sua vez** são divididos em passos, cada um com a sua resposta oculta.

**Antes de começar:** abra o ambiente do curso (rotina do Módulo 01) e abra este notebook, `04-seletores-xpath-css.ipynb`. Os exemplos usam arquivos dos módulos anteriores: o site da Livraria Exemplo (Módulo 02) e o catálogo de livros baixado na missão do Módulo 03.

▶️ Comece importando as bibliotecas usadas no módulo. O `Selector` é a peça central do parsel.

In [ ]:
import time
from pathlib import Path

import requests
from parsel import Selector

---

# Parte 1: Do texto à árvore

## 1.1 O Selector

Todo scraper com o parsel segue o mesmo caminho: o texto do HTML vira uma árvore, uma busca na árvore devolve uma lista de elementos, e dessa lista saem os dados:

<img src="imagens/do-texto-a-arvore.png" alt="Quatro etapas: o HTML como texto, o Selector com a árvore, a busca com xpath que devolve uma lista, e os dados com get e getall" width="1120">

Os primeiros exemplos usam a página da Livraria Exemplo, do Módulo 02: ela é pequena, e você já conhece a árvore dela.

▶️ Leia o arquivo `index.html` da Livraria Exemplo. O resultado é texto (`str`):

In [ ]:
html = Path("../02-fundamentos-da-web/site-exemplo/index.html").read_text(encoding="utf-8")
type(html)

▶️ Entregue o texto ao `Selector`. A variável `pagina` passa a guardar a **árvore** da página:

In [ ]:
pagina = Selector(text=html)
pagina

O resumo mostra o começo do documento. A partir de agora, buscas são feitas na árvore com o método `xpath()`, que recebe o caminho até o elemento.

▶️ Procure o título principal da página, o elemento `h1`:

In [ ]:
pagina.xpath("//h1")

O resultado é uma **lista** (com um item só, entre colchetes), e cada item é um `Selector` que aponta para um elemento encontrado.

▶️ O método `get()` tira o primeiro item da lista, como texto:

In [ ]:
pagina.xpath("//h1").get()

Veio o elemento inteiro, com as tags. Para ter só o texto de dentro dele, o caminho termina em `/text()`.

▶️ Execute:

In [ ]:
pagina.xpath("//h1/text()").get()

## 1.2 Sempre uma lista: get() e getall()

Um XPath descreve um **padrão**, e uma página pode ter vários elementos que o seguem. Por isso, o resultado de `xpath()` é sempre uma lista, mesmo quando só um elemento combina.

▶️ Procure todos os preços. O XPath `//span[@class="preco"]` encontra os elementos `span` cuja classe é `preco` (a parte entre colchetes é uma condição, assunto da seção 2.2):

In [ ]:
precos = pagina.xpath('//span[@class="preco"]/text()')
len(precos)

São 4 preços, um por livro.

▶️ O método `getall()` devolve **todos** os resultados, em uma lista de textos:

In [ ]:
precos.getall()

▶️ E o `get()` devolve só o **primeiro**:

In [ ]:
precos.get()

▶️ Quando nada é encontrado, a lista fica vazia. Procure uma tabela, que essa página não tem:

In [ ]:
pagina.xpath("//table").getall()

▶️ Nesse caso, o `get()` devolve `None` ("nada"). Para mostrar o resultado, use `print()`, já que o notebook não exibe `None` sozinho:

In [ ]:
print(pagina.xpath("//table").get())

▶️ O `get()` aceita um valor padrão, devolvido quando nada é encontrado:

In [ ]:
pagina.xpath("//table").get(default="não encontrado")

### Sua vez: as primeiras buscas

**Passo 1 de 3.** Mostre o texto de **todos** os títulos de livros. Eles são links (`a`) com a classe `titulo`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
pagina.xpath('//a[@class="titulo"]/text()').getall()
```
</details>

**Passo 2 de 3.** Mostre só o **primeiro** autor. Os autores são `span` com a classe `autor`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
pagina.xpath('//span[@class="autor"]/text()').get()
```
</details>

**Passo 3 de 3.** Quantos autores há na página?

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
len(pagina.xpath('//span[@class="autor"]'))
```
</details>

> **Referências da Parte 1:** [Usage](https://parsel.readthedocs.io/en/latest/usage.html) da documentação do parsel, seções *Creating selectors* e *Using selectors* (em inglês).

---

# Parte 2: Escrevendo XPaths

## 2.1 Caminhos: / e //

Um XPath é lido da esquerda para a direita, um passo de cada vez, como um caminho de pastas:

<img src="imagens/xpath-anatomia.png" alt="O XPath //ul[@class='livros']/li/span[@class='preco']/text() dividido em partes coloridas, com a explicação de / e //" width="1120">

▶️ Um caminho que começa com `/` parte da raiz e desce **um nível por passo**. É preciso escrever cada nível:

In [ ]:
pagina.xpath("/html/body/header/h1/text()").get()

▶️ Com `//`, os níveis do meio podem ser pulados: `//h1` procura o `h1` em qualquer lugar da página.

In [ ]:
pagina.xpath("//h1/text()").get()

A diferença também vale no meio do caminho. Os links dos livros estão dentro de `main`, mas não são filhos diretos dele: há uma lista (`ul`) e os itens (`li`) no meio.

▶️ Com `/`, procurando só os filhos diretos de `main`, nada é encontrado:

In [ ]:
pagina.xpath("//main/a").getall()

▶️ Com `//`, procurando em qualquer nível abaixo de `main`, os 4 links aparecem:

In [ ]:
len(pagina.xpath("//main//a"))

## 2.2 Predicados: condições entre colchetes

Um **predicado** é uma condição escrita entre colchetes, logo depois de um passo. Só os elementos que cumprem a condição continuam no caminho.

> 💡 No Python, o XPath é um texto. Como o XPath usa aspas duplas nos valores (`@class="autor"`), escreva o texto do Python entre **aspas simples**: `'//span[@class="autor"]'`.

▶️ `[@class="autor"]`: o atributo `class` precisa ser igual a `autor`. O `@` indica um atributo.

In [ ]:
pagina.xpath('//span[@class="autor"]/text()').getall()

▶️ `[1]`: a **posição**. Pegue o título do primeiro livro da lista (no XPath, a contagem começa em 1):

In [ ]:
pagina.xpath('//li[@class="livro"][1]/a/text()').get()

▶️ `[last()]`: o **último**. Pegue o título do último livro:

In [ ]:
pagina.xpath('//li[@class="livro"][last()]/a/text()').get()

▶️ `[text()="..."]`: o texto do elemento precisa ser exatamente igual. Encontre o link do livro *Iracema*:

In [ ]:
pagina.xpath('//a[text()="Iracema"]').get()

▶️ `contains()`: o texto (ou atributo) precisa **conter** um trecho. Encontre os autores com `Machado` no nome:

In [ ]:
pagina.xpath('//span[contains(text(), "Machado")]/text()').getall()

O `contains()` também é a forma de lidar com classes de **várias palavras**. No books.toscrape.com, por exemplo, a nota de cada livro tem `class="star-rating Three"`: o XPath `[@class="star-rating"]` não encontra nada, porque a classe não é exatamente esse texto, mas `[contains(@class, "star-rating")]` encontra. Você vai usar isso na Parte 5.

### Sua vez: predicados

**Passo 1 de 3.** Mostre o **preço** do **último** livro.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
pagina.xpath('//li[@class="livro"][last()]/span[@class="preco"]/text()').get()
```
</details>

**Passo 2 de 3.** Mostre o título do **segundo** livro.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
pagina.xpath('//li[@class="livro"][2]/a/text()').get()
```
</details>

**Passo 3 de 3.** Mostre os títulos que contêm a palavra `Memórias`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
pagina.xpath('//a[contains(text(), "Memórias")]/text()').getall()
```
</details>

## 2.3 O que extrair: texto, atributo ou o elemento

O final do XPath escolhe **o que** volta de cada elemento encontrado:

<img src="imagens/o-que-extrair.png" alt="Tabela com os finais de XPath: o elemento inteiro, text(), @href, string() e normalize-space(), com o resultado de cada um" width="1120">

▶️ Sem final especial, volta o **elemento inteiro**, com as tags:

In [ ]:
pagina.xpath('//a[@class="titulo"]').get()

▶️ Com `/text()`, só o **texto**:

In [ ]:
pagina.xpath('//a[@class="titulo"]/text()').get()

▶️ Com `/@href`, o valor do **atributo** `href`: o endereço para onde o link leva. Pegue todos:

In [ ]:
pagina.xpath('//a[@class="titulo"]/@href').getall()

O `text()` pega só o texto que está **diretamente** dentro do elemento. Um item da lista (`li`) não tem texto direto: o título, o autor e o preço estão em elementos filhos.

▶️ A função `string()` junta todo o texto do elemento **e dos seus descendentes**:

In [ ]:
pagina.xpath('string(//li[@class="livro"][1])').get()

O texto veio, mas com as quebras de linha (`\n`) e os espaços da formatação do HTML.

▶️ A função `normalize-space()` faz o mesmo que o `string()` e ainda limpa os espaços sobrando:

In [ ]:
pagina.xpath('normalize-space(//li[@class="livro"][1])').get()

### Sua vez: o que extrair

**Passo 1 de 3.** Mostre o valor do atributo `class` do elemento `ul` da página.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
pagina.xpath('//ul/@class').get()
```
</details>

**Passo 2 de 3.** Mostre o texto do rodapé (`footer`), limpo, com `normalize-space()`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
pagina.xpath('normalize-space(//footer)').get()
```
</details>

**Passo 3 de 3.** Mostre o endereço (`href`) do link do livro *Dom Casmurro*. Dica: combine o predicado de texto com o atributo.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
pagina.xpath('//a[text()="Dom Casmurro"]/@href').get()
```
</details>

> **Referências da Parte 2:** [XPath](https://developer.mozilla.org/en-US/docs/Web/XML/XPath) e [XPath functions](https://developer.mozilla.org/en-US/docs/Web/XML/XPath/Reference/Functions) (MDN, em inglês); [XML Path Language (XPath) 1.0](https://www.w3.org/TR/xpath-10/), a especificação do W3C (em inglês).

---

# Parte 3: Eixos, navegando a partir de um elemento

## 3.1 Pai, irmãos e ancestrais

Até aqui, os caminhos só **desciam** pela árvore. Os **eixos** permitem ir em outras direções a partir de um elemento: subir para o pai, ir para o irmão anterior ou para o seguinte. Eles usam o vocabulário de família do Módulo 02:

<img src="imagens/eixos.png" alt="A árvore da Livraria Exemplo com o autor de Iracema como ponto de partida e o pai, os irmãos e os ancestrais marcados, e uma tabela com o XPath de cada eixo" width="1120">

O ponto de partida dos exemplos é o autor do livro *Iracema*. O XPath dele fica guardado em uma variável, para que os exemplos acrescentem só o eixo.

▶️ Guarde o XPath do ponto de partida e confira o que ele encontra:

In [ ]:
autor = '//span[text()="José de Alencar"]'
pagina.xpath(autor).get()

▶️ **Pai** (`..`): suba um nível, até o `li` que contém o autor, e veja a classe dele:

In [ ]:
pagina.xpath(autor + "/../@class").get()

▶️ **Irmão anterior** (`preceding-sibling::`): o elemento `a` que vem antes do autor, dentro do mesmo `li`, é o título:

In [ ]:
pagina.xpath(autor + "/preceding-sibling::a/text()").get()

▶️ **Irmão seguinte** (`following-sibling::`): o `span` que vem depois do autor é o preço:

In [ ]:
pagina.xpath(autor + "/following-sibling::span/text()").get()

▶️ **Ancestral** (`ancestor::`): qualquer elemento acima, na mesma linha da árvore. Suba até o `ul` e veja a classe dele:

In [ ]:
pagina.xpath(autor + "/ancestor::ul/@class").get()

Os eixos se combinam com tudo o que você já viu.

▶️ Os títulos dos livros de Machado de Assis: encontre os autores com esse nome e, de cada um, vá para o irmão anterior:

In [ ]:
pagina.xpath('//span[text()="Machado de Assis"]/preceding-sibling::a/text()').getall()

### Sua vez: eixos

**Passo 1 de 3.** Mostre o **autor** do livro *O Cortiço*: parta do título (`//a[text()="O Cortiço"]`) e vá para o irmão seguinte.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
pagina.xpath('//a[text()="O Cortiço"]/following-sibling::span[@class="autor"]/text()').get()
```
</details>

**Passo 2 de 3.** Mostre o **preço** do mesmo livro, partindo do título.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
pagina.xpath('//a[text()="O Cortiço"]/following-sibling::span[@class="preco"]/text()').get()
```
</details>

**Passo 3 de 3.** Mostre o **título** do livro que custa `R$ 32,00`, partindo do preço.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
pagina.xpath('//span[text()="R$ 32,00"]/preceding-sibling::a/text()').get()
```
</details>

## 3.2 Rótulo e valor: os irmãos em uma página real

Muitas páginas mostram dados em tabelas de duas colunas: o **rótulo** à esquerda e o **valor** à direita. Os irmãos resolvem esse caso com elegância: procure o rótulo pelo texto e pegue o irmão seguinte.

<img src="imagens/rotulo-valor.png" alt="A tabela Product Information da página de um livro, com a linha Availability destacada, o HTML da linha e os três passos do XPath" width="1120">

▶️ Baixe a página do livro *A Light in the Attic* com o requests, como no Módulo 03, e crie o `Selector`:

In [ ]:
resposta = requests.get("https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html", timeout=20)
resposta.encoding = "utf-8"
pagina_livro = Selector(text=resposta.text)

▶️ Veja todos os rótulos da tabela. Eles ficam nos elementos `th`:

In [ ]:
pagina_livro.xpath("//table//th/text()").getall()

▶️ Pegue a disponibilidade: encontre o `th` com o texto `Availability` e vá para o `td` irmão seguinte:

In [ ]:
pagina_livro.xpath('//th[text()="Availability"]/following-sibling::td/text()').get()

▶️ O mesmo XPath serve para qualquer rótulo, trocando só o texto. Pegue o código do produto (`UPC`):

In [ ]:
pagina_livro.xpath('//th[text()="UPC"]/following-sibling::td/text()').get()

Um cuidado com tabelas: ao inspecionar esta tabela no DevTools, você vai ver um elemento `<tbody>` entre o `table` e as linhas (`tr`). Mas ele **não está no HTML** que o servidor enviou: o navegador o acrescenta sozinho ao montar o DOM.

▶️ Um XPath com `tbody`, copiado do navegador, não encontra nada no Python:

In [ ]:
len(pagina_livro.xpath("//table/tbody/tr"))

▶️ Com `//` entre o `table` e o `tr`, o XPath funciona nos dois casos, com ou sem `tbody`:

In [ ]:
len(pagina_livro.xpath("//table//tr"))

### Sua vez: rótulo e valor

**Passo 1 de 4.** Mostre o número de avaliações do livro: o rótulo é `Number of reviews`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
pagina_livro.xpath('//th[text()="Number of reviews"]/following-sibling::td/text()').get()
```
</details>

**Passo 2 de 4.** Mostre o preço com impostos: o rótulo é `Price (incl. tax)`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
pagina_livro.xpath('//th[text()="Price (incl. tax)"]/following-sibling::td/text()').get()
```
</details>

**Passo 3 de 4.** Na parte de cima da página, o título do livro (`h1`) é o **irmão anterior** do preço (`p` com a classe `price_color`). Os dois ficam em um `div` cuja classe **contém** `product_main`. Pegue o título partindo do preço.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
pagina_livro.xpath('//div[contains(@class, "product_main")]/p[@class="price_color"]/preceding-sibling::h1/text()').get()
```
</details>

**Passo 4 de 4.** A categoria do livro está na trilha de navegação no topo da página (`ul` com a classe `breadcrumb`): é o link do **terceiro** item (`li`). Mostre a categoria.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
pagina_livro.xpath('//ul[@class="breadcrumb"]/li[3]/a/text()').get()
```
</details>

> **Referências da Parte 3:** [Axes](https://developer.mozilla.org/en-US/docs/Web/XML/XPath/Reference/Axes) (MDN, em inglês); seção *Location Paths* da [especificação do XPath 1.0](https://www.w3.org/TR/xpath-10/) (em inglês).

---

# Parte 4: Do navegador ao código

## 4.1 O caminho até um bom XPath

Em um site novo, você não conhece a árvore. O caminho para chegar a um XPath confiável é sempre o mesmo:

1. **Selecione o dado na página** com o seletor de elementos do DevTools (seção 4.2). O Inspetor mostra o elemento na árvore.
2. **Entenda a árvore em volta.** Repare nos pais, nos irmãos e nas classes do elemento. Passe o mouse sobre as linhas do Inspetor: a parte correspondente da página fica destacada.
3. **Procure pontos de apoio estáveis.** Classes com nomes que descrevem o conteúdo (`price_color`, `product_pod`), atributos (`title`, `href`) e textos de rótulos (`Availability`) são bons pontos de apoio. Posições (`[2]`) e longos caminhos a partir da raiz não são.
4. **Teste o XPath no navegador** e confira **quantos resultados** ele encontra: na busca do Inspetor (seção 4.3) ou no console (seção 4.4).
5. **Leve o XPath para o Python** e confira o resultado com `get()` ou `getall()`.

## 4.2 Selecionando o elemento na página

O botão com a seta, no canto esquerdo da barra do DevTools, liga o **seletor de elementos**. Com ele ligado, cada elemento sob o mouse fica destacado na página, e um clique o seleciona no Inspetor. O atalho é **`Ctrl + Shift + C`** (no macOS, `Cmd + Shift + C`), no Firefox, no Edge e no Chrome.

Abra a página do livro *[A Light in the Attic](https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)*, aperte `F12`, ligue o seletor e passe o mouse sobre a disponibilidade, na tabela *Product Information*:

<img src="imagens/seletor-elemento.png" alt="O Firefox com o seletor de elementos ligado: o mouse sobre a célula In stock (22 available), uma etiqueta td 411.767 x 37 sobre ela, e a linha correspondente marcada no Inspetor" width="1130">

- ① O botão do **seletor de elementos**, ligado (em azul).
- ② O elemento sob o mouse, destacado na página. A etiqueta escura mostra a tag (`td`) e o tamanho do elemento.
- ③ A linha correspondente no Inspetor: `<td>In stock (22 available)</td>`. Ao clicar, ela fica selecionada.

Repare na árvore em volta da linha ③: o `td` com o valor tem um irmão anterior, `<th>Availability</th>`, e os dois estão dentro de um `tr`. É exatamente o par **rótulo e valor** da seção 3.2.

## 4.3 Testando o XPath na busca do Inspetor: quantos resultados?

O campo de busca do Inspetor (*Search HTML*, no topo do painel) aceita **XPath**, além de texto comum. Ao digitar um XPath e apertar `Enter`, o Inspetor seleciona o primeiro elemento encontrado e mostra a **contagem**: `1 of 7` quer dizer "este é o 1º de 7 resultados". As setas ao lado da contagem percorrem os outros resultados.

Essa contagem é o melhor teste de um XPath. Veja dois XPaths para a disponibilidade, na mesma página:

<img src="imagens/busca-xpath.png" alt="Duas capturas do Firefox: o XPath //table//td encontra 7 células, todas destacadas na tabela; o XPath com o rótulo Availability e following-sibling encontra só a célula da disponibilidade" width="1110">

O primeiro XPath **parou em um galho anterior**: a tabela, que se divide em 7 linhas, cada uma com uma célula de valor. Ele não está errado como caminho, mas leva a 7 dados, e você quer um. O segundo **desce até o último galho**, usando o rótulo como ponto de apoio, e leva exatamente ao dado: 1 resultado.

A regra vale para qualquer busca: **o número de resultados deve ser igual ao número de dados que você quer**.

- Para **um dado** (o título de um livro, a disponibilidade), o XPath certo encontra **1** elemento. Se encontrar vários, desça mais na árvore, ou acrescente uma condição.
- Para uma **lista** (os 20 livros da página), o XPath certo encontra **um elemento por item**: 20. Se encontrar mais (por exemplo, `//article//p`, que pega todos os parágrafos de dentro dos livros), ele está pegando pedaços demais; se encontrar só 1 (por exemplo, `//ol`, a lista que guarda todos os livros), parou em um galho acima dos itens.

> 💡 No Edge e no Chrome, a busca do painel Elementos abre com **`Ctrl + F`** e também aceita XPath: o campo diz *Encontrar por cadeia de caracteres, seletor ou XPath*. A contagem aparece no mesmo formato, como `1 de 7`.

▶️ No Python, a contagem é o `len()` do resultado. Confira o primeiro XPath na página do livro, baixada na seção 3.2:

In [ ]:
len(pagina_livro.xpath("//table//td"))

▶️ E o segundo, que desce até o último galho:

In [ ]:
len(pagina_livro.xpath('//th[text()="Availability"]/following-sibling::td'))

▶️ Com o XPath certo, o `get()` devolve o dado que você queria, sem ambiguidade:

In [ ]:
pagina_livro.xpath('//th[text()="Availability"]/following-sibling::td/text()').get()

### Sua vez: o último galho

**Passo 1 de 3.** Na página do livro, o preço aparece em dois lugares: em destaque, ao lado do título, e na tabela. Conte quantos elementos `p` com a classe `price_color` existem na página. Teste também na busca do Inspetor: `//p[@class="price_color"]`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
len(pagina_livro.xpath('//p[@class="price_color"]'))
```
</details>

**Passo 2 de 3.** Agora conte as células `td` que contêm o símbolo `£`. Quantas são, e o que elas representam?

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
len(pagina_livro.xpath('//td[contains(text(), "£")]'))
```
</details>

**Passo 3 de 3.** Escreva um XPath que encontre **só** o valor do imposto (rótulo `Tax`) e confira que ele dá exatamente 1 resultado.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
len(pagina_livro.xpath('//th[text()="Tax"]/following-sibling::td'))
```
</details>

O `//p[@class="price_color"]` dá 1 resultado: o preço em destaque é único na página. Já as células com `£` são 3 (o preço sem imposto, com imposto e o imposto): um caminho que leva a 3 dados, quando você quer um. O rótulo é o que leva ao último galho.

## 4.4 Testando um XPath no console

O console do DevTools tem a função `$x()`, que recebe um XPath e mostra os elementos que ele encontra, na própria página. É a forma mais rápida de testar um XPath antes de levá-lo para o código. Ela existe no Firefox, no Edge e no Chrome.

Abra **[books.toscrape.com](https://books.toscrape.com)**, aperte `F12`, vá para a aba **Console** e digite os XPaths da imagem:

<img src="imagens/console-xpath.png" alt="O Firefox com o console à direita: o XPath dos preços devolve uma lista com 20 elementos, e o XPath dos blocos de livro devolve outra lista com 20; três preços estão destacados na página" width="1130">

- ① A aba **Console**.
- ② O XPath dos preços, escrito com `$x('...')`. No console, use aspas simples por fora e duplas por dentro, como no Python.
- ③ O resultado: `Array(20)`, uma lista com os 20 preços da página. Passando o mouse sobre cada item, o elemento correspondente fica destacado na página, como os três preços marcados.
- ④ O XPath do contêiner de cada livro, o `article` com a classe `product_pod`.
- ⑤ Outra lista com 20 itens: um bloco por livro. É a mesma lista que o parsel vai devolver.

> 💡 Se o console mostrar um aviso sobre colar código antes de deixar você digitar, siga a instrução da mensagem: é uma proteção do navegador contra golpes.

## 4.5 Copiar o XPath ou escrever o seu?

O DevTools oferece a opção **Copiar → XPath** (botão direito sobre o elemento no painel de elementos). Ela é útil para descobrir onde o elemento está, mas o XPath copiado descreve **posições**, e não o significado do elemento:

<img src="imagens/copiado-vs-escrito.png" alt="Comparação entre o XPath absoluto copiado do Firefox, cheio de posições, e o XPath escrito com a classe price_color, que encontra os 20 preços" width="1120">

No Firefox, o caminho copiado parte da raiz, como na imagem. No Edge e no Chrome, ele costuma começar pelo `id` do ancestral mais próximo que tenha um, como `//*[@id="default"]/div/...`, mas continua cheio de posições.

A partir daqui, os exemplos usam o **catálogo salvo na missão do Módulo 03**, na pasta `03-requests/dados/catalogo`.

▶️ Liste os arquivos do catálogo:

In [ ]:
pasta_catalogo = Path("../03-requests/dados/catalogo")
arquivos = sorted(pasta_catalogo.glob("*.html"))
len(arquivos)

Devem aparecer 50 arquivos. Se apareceu `0`, a missão do Módulo 03 não foi feita neste computador.

▶️ A célula abaixo resolve isso: se o catálogo não existir, ela baixa as 50 páginas (cerca de 40 segundos). Se ele já existir, ela não faz nada.

In [ ]:
if not arquivos:
    print("Catálogo não encontrado. Baixando as 50 páginas...")
    pasta_catalogo.mkdir(parents=True, exist_ok=True)
    for numero in range(1, 51):
        resposta = requests.get(f"https://books.toscrape.com/catalogue/page-{numero}.html", timeout=20)
        resposta.encoding = "utf-8"
        (pasta_catalogo / f"pagina-{numero:02d}.html").write_text(resposta.text, encoding="utf-8")
        time.sleep(0.5)
    arquivos = sorted(pasta_catalogo.glob("*.html"))
len(arquivos)

▶️ Crie o `Selector` da primeira página do catálogo, lida do disco:

In [ ]:
primeira = Selector(text=arquivos[0].read_text(encoding="utf-8"))

▶️ Teste o XPath copiado do Firefox. Ele funciona, mas encontra **um** preço:

In [ ]:
copiado = "/html/body/div/div/div/div/section/div[2]/ol/li[1]/article/div[2]/p[1]/text()"
primeira.xpath(copiado).getall()

▶️ Agora o XPath escrito com a classe. Ele encontra **todos** os preços da página:

In [ ]:
escrito = '//p[@class="price_color"]/text()'
len(primeira.xpath(escrito))

Os dois funcionam hoje, mas o copiado depende de cada `div` do caminho continuar exatamente onde está. Se o site ganhar um aviso no topo, ele quebra; o escrito continua funcionando.

---

# Parte 5: Contêineres e listas de itens

## 5.1 O contêiner: um galho por item

Em uma página com muitos itens iguais (livros, produtos, citações, linhas de uma tabela), cada item fica dentro de um bloco com a mesma estrutura: o **contêiner**. O jeito confiável de extrair os dados é em duas etapas:

<img src="imagens/conteiner.png" alt="Os 20 blocos de livros da página, o XPath do contêiner que devolve uma lista com 20 itens, e os XPaths relativos com ponto usados dentro de cada item" width="1120">

1. Um XPath encontra **todos os contêineres**: o resultado é uma lista, com um item por livro.
2. Dentro de **cada** contêiner, XPaths **relativos** (começando com ponto) buscam os dados daquele item.

Assim, o título, o preço e a nota de um livro nunca se misturam com os de outro.

▶️ Encontre os contêineres de livro da primeira página:

In [ ]:
livros = primeira.xpath('//article[@class="product_pod"]')
len(livros)

São 20 contêineres, um por livro. Cada item da lista é um `Selector` próprio, onde dá para continuar a busca.

▶️ Pegue o primeiro livro da lista:

In [ ]:
primeiro = livros[0]

▶️ Busque o título **dentro dele**. O XPath começa com `.`, que significa "a partir deste elemento":

In [ ]:
primeiro.xpath(".//h3/a/@title").get()

O título completo fica no atributo `title` do link; o texto visível é encurtado (`A Light in the ...`), como você viu no exercício do DevTools no Módulo 02.

O ponto no início faz toda a diferença. Sem ele, o XPath volta para o topo da página, mesmo quando chamado a partir de um livro.

▶️ Veja o erro acontecer: o XPath **sem ponto**, chamado em cada um dos três primeiros livros, devolve sempre o mesmo título:

In [ ]:
for livro in livros[:3]:
    print(livro.xpath("//h3/a/@title").get())

▶️ Agora **com ponto**. Cada livro devolve o seu próprio título:

In [ ]:
for livro in livros[:3]:
    print(livro.xpath(".//h3/a/@title").get())

## 5.2 Extraindo os dados de um livro

Com o primeiro livro em mãos, cada dado é extraído e testado separadamente, antes de juntar tudo.

▶️ O preço:

In [ ]:
primeiro.xpath('.//p[@class="price_color"]/text()').get()

▶️ A nota fica na classe do parágrafo de estrelas, que tem duas palavras. Por isso, o predicado usa `contains()`:

In [ ]:
primeiro.xpath('./p[contains(@class, "star-rating")]/@class').get()

A classe é `star-rating Three`: a nota é a última palavra.

▶️ O método `split()` do Python separa as palavras, e `[-1]` pega a última:

In [ ]:
primeiro.xpath('./p[contains(@class, "star-rating")]/@class').get().split()[-1]

▶️ A disponibilidade. Com `text()`, o resultado vem cheio de espaços e quebras de linha, porque há um ícone no meio do parágrafo:

In [ ]:
primeiro.xpath('.//p[contains(@class, "availability")]/text()').getall()

▶️ Com `normalize-space()`, fica limpo:

In [ ]:
primeiro.xpath('normalize-space(.//p[contains(@class, "availability")])').get()

▶️ O link para a página do livro, no atributo `href`:

In [ ]:
primeiro.xpath(".//h3/a/@href").get()

O link é **relativo**: não começa com `https://`. No Módulo 05, ele será completado para levar à página do livro.

▶️ Junte tudo em uma **função** que recebe um contêiner de livro e devolve um dicionário com os dados:

In [ ]:
def extrair_livro(livro):
    return {
        "titulo": livro.xpath(".//h3/a/@title").get(),
        "preco": livro.xpath('.//p[@class="price_color"]/text()').get(),
        "nota": livro.xpath('./p[contains(@class, "star-rating")]/@class').get().split()[-1],
        "estoque": livro.xpath('normalize-space(.//p[contains(@class, "availability")])').get(),
        "link": livro.xpath(".//h3/a/@href").get(),
    }

▶️ Teste a função com o primeiro livro:

In [ ]:
extrair_livro(primeiro)

▶️ E com o último livro da página:

In [ ]:
extrair_livro(livros[-1])

### Sua vez: as citações do quotes.toscrape.com

**Passo 1 de 5.** Baixe a página `https://quotes.toscrape.com/` com o requests e crie um `Selector` chamado `pagina_citacoes`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
resposta = requests.get("https://quotes.toscrape.com/", timeout=20)
pagina_citacoes = Selector(text=resposta.text)
```
</details>

**Passo 2 de 5.** Inspecione uma citação no navegador: cada uma fica em um `div` com a classe `quote`. Encontre todos esses contêineres e guarde em `blocos`. Quantos são?

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
blocos = pagina_citacoes.xpath('//div[@class="quote"]')
len(blocos)
```
</details>

**Passo 3 de 5.** Dentro do **primeiro** bloco, pegue o texto da citação (`span` com a classe `text`).

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
blocos[0].xpath('./span[@class="text"]/text()').get()
```
</details>

**Passo 4 de 5.** Ainda no primeiro bloco, pegue o autor (`small` com a classe `author`) e a lista de todas as tags (links `a` com a classe `tag`).

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
print(blocos[0].xpath('.//small[@class="author"]/text()').get())
print(blocos[0].xpath('.//a[@class="tag"]/text()').getall())
```
</details>

**Passo 5 de 5.** Percorra todos os blocos com um `for` e mostre o autor de cada citação.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
for bloco in blocos:
    print(bloco.xpath('.//small[@class="author"]/text()').get())
```
</details>

## 5.3 Missão: os 1000 livros

Agora, o catálogo inteiro: para cada um dos 50 arquivos, crie o `Selector`, encontre os 20 contêineres e extraia os dados de cada livro com a função `extrair_livro()`.

▶️ Crie uma lista vazia, que vai receber um dicionário por livro:

In [ ]:
todos = []

▶️ Percorra os arquivos. O `for` de dentro percorre os livros de cada página, e o `append()` acrescenta cada dicionário à lista:

In [ ]:
for arquivo in arquivos:
    pagina_catalogo = Selector(text=arquivo.read_text(encoding="utf-8"))
    for livro in pagina_catalogo.xpath('//article[@class="product_pod"]'):
        todos.append(extrair_livro(livro))

▶️ Confira quantos livros foram extraídos:

In [ ]:
len(todos)

▶️ Veja os três primeiros:

In [ ]:
todos[:3]

▶️ E o último:

In [ ]:
todos[-1]

▶️ Uma conferência rápida: quantos livros têm nota máxima (`Five`)?

In [ ]:
cinco_estrelas = 0
for item in todos:
    if item["nota"] == "Five":
        cinco_estrelas += 1
cinco_estrelas

Os 1000 livros estão em uma lista de dicionários, extraídos em poucos segundos dos arquivos salvos, sem nenhum pedido ao site. Repare que os dados ainda estão como **texto**: o preço é `"£51.77"`, e a nota, `"Three"`. Transformá-los em números e montar uma tabela é o assunto do Módulo 06.

### Sua vez: a tabela de times

**Passo 1 de 5.** Baixe a página `https://www.scrapethissite.com/pages/forms/` e crie um `Selector` chamado `pagina_times`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
resposta = requests.get("https://www.scrapethissite.com/pages/forms/", timeout=20)
pagina_times = Selector(text=resposta.text)
```
</details>

**Passo 2 de 5.** Cada time é uma linha da tabela: um `tr` com a classe `team`. Encontre os contêineres e guarde em `times`. Quantos são?

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
times = pagina_times.xpath('//tr[@class="team"]')
len(times)
```
</details>

**Passo 3 de 5.** No primeiro time, a célula do nome é um `td` com a classe `name`. Pegue o texto com `text()` e veja os espaços que sobram.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
times[0].xpath('./td[@class="name"]/text()').get()
```
</details>

**Passo 4 de 5.** Pegue o mesmo nome, agora limpo, com `normalize-space()`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
times[0].xpath('normalize-space(./td[@class="name"])').get()
```
</details>

**Passo 5 de 5.** Crie uma lista de dicionários com o nome, o ano (`td` com a classe `year`) e as vitórias (`td` com a classe `wins`) de cada time da página, e mostre os dois primeiros.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
lista_times = []
for time_ in times:
    lista_times.append({
        "nome": time_.xpath('normalize-space(./td[@class="name"])').get(),
        "ano": time_.xpath('normalize-space(./td[@class="year"])').get(),
        "vitorias": time_.xpath('normalize-space(./td[@class="wins"])').get(),
    })
lista_times[:2]
```
</details>

A variável do laço se chama `time_`, com o sublinhado no final, porque `time` já é o nome do módulo importado no início do notebook.

> **Referências da Parte 5:** seção *Working with relative XPaths* da documentação do [parsel](https://parsel.readthedocs.io/en/latest/usage.html) (em inglês), que mostra o mesmo erro do XPath sem ponto; [Web Console Helpers](https://firefox-source-docs.mozilla.org/devtools-user/web_console/helpers/), com a função `$x()` no Firefox, e a [referência da API Console Utilitários](https://developer.chrome.com/docs/devtools/console/utilities?hl=pt-br), no Chrome e no Edge.

---

# Parte 6: Seletores CSS

## 6.1 A mesma busca, em outra linguagem

O parsel também aceita **seletores CSS**, a mesma linguagem usada para dar estilo às páginas. Para os casos simples, eles são mais curtos que o XPath:

| O que buscar | XPath | CSS |
|---|---|---|
| Elementos `p` com a classe `price_color` | `//p[@class="price_color"]` | `p.price_color` |
| O texto desses elementos | `//p[@class="price_color"]/text()` | `p.price_color::text` |
| O atributo `href` dos links dentro de `h3` | `//h3/a/@href` | `h3 a::attr(href)` |
| Filho direto | `//article/h3` | `article > h3` |
| Elemento com um `id` | `//*[@id="default"]` | `#default` |

No CSS, o ponto (`.`) indica uma classe, o espaço indica "em qualquer nível abaixo", e o `::text` e o `::attr()` são extensões do parsel para extrair texto e atributos.

▶️ Os três primeiros preços da primeira página do catálogo, com CSS. O método é o `css()`:

In [ ]:
primeira.css("p.price_color::text").getall()[:3]

▶️ Um atributo, com `::attr()`. O título do primeiro livro:

In [ ]:
primeira.css("h3 a::attr(title)").get()

No CSS, `.star-rating` encontra elementos que **têm** essa classe, mesmo que tenham outras: não é preciso o `contains()`.

▶️ A classe da nota do primeiro livro:

In [ ]:
primeira.css("p.star-rating::attr(class)").get()

▶️ Os dois podem ser combinados: encontre os contêineres com CSS e busque dentro deles com XPath:

In [ ]:
primeira.css("article.product_pod")[0].xpath(".//h3/a/@title").get()

## 6.2 Por que o curso usa XPath

O CSS não consegue **subir** na árvore (não há pai nem ancestral), não consegue ir para o **irmão anterior** e não consegue escolher um elemento pelo **texto** dele. Por isso, o caso do rótulo e valor da seção 3.2 não tem solução só com CSS.

O XPath faz tudo o que o CSS faz e mais. Por isso ele é a ferramenta principal do curso, e o CSS fica como um atalho para os casos simples. No Módulo 10, você vai reencontrar os dois no Scrapy, que usa o mesmo parsel por dentro.

### Sua vez: traduzindo para CSS

**Passo 1 de 2.** Com CSS, pegue o texto de todos os títulos de livros da Livraria Exemplo (a variável `pagina`, da Parte 1). Eles são links com a classe `titulo`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
pagina.css("a.titulo::text").getall()
```
</details>

**Passo 2 de 2.** Com CSS, pegue os endereços (`href`) desses mesmos links.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
pagina.css("a.titulo::attr(href)").getall()
```
</details>

> **Referências da Parte 6:** seção *Using selectors* da documentação do [parsel](https://parsel.readthedocs.io/en/latest/usage.html), com as extensões `::text` e `::attr()` (em inglês); [Seletores CSS](https://developer.mozilla.org/pt-BR/docs/Web/CSS/Guides/Selectors) (MDN).

---

## Problemas comuns e como resolver

| Sintoma | Causa provável | Solução |
|---|---|---|
| `get()` devolve `None` | O XPath não encontrou nada | Teste o XPath na busca do Inspetor ou no console com `$x()`; confira o nome das tags e das classes, letra por letra |
| O XPath encontra mais elementos do que deveria | Ele parou em um galho anterior, que se divide em vários | Desça até o último galho, ou acrescente uma condição (classe, rótulo). Confira a contagem na busca do Inspetor (seção 4.3) |
| `[@class="..."]` não encontra um elemento que tem essa classe | A classe tem várias palavras (`star-rating Three`) | Use `[contains(@class, "star-rating")]` ou o CSS `.star-rating` |
| Todos os itens de um laço devolvem o mesmo valor | XPath sem o ponto inicial dentro do contêiner | Use `.//` ou `./` nos XPaths de dentro do laço (seção 5.1) |
| O texto vem com `\n` e muitos espaços | Formatação do HTML | Use `normalize-space(...)` |
| `text()` não traz o texto que aparece na tela | O texto está em elementos filhos | Use `string(...)` ou `normalize-space(...)`, ou desça até o filho certo |
| O XPath funciona no console do navegador, mas não no Python | O DOM do navegador é diferente do HTML recebido: `<tbody>` acrescentado, ou conteúdo criado por JavaScript | Troque `/tbody/` por `//`; confira o HTML recebido com **Exibir código-fonte** (`Ctrl + U`) |
| `ValueError: XPath error: Invalid expression` | Erro de escrita no XPath: colchete, aspas ou parêntese faltando | Confira os pares de `[ ]`, `( )` e aspas. No Python, use aspas simples por fora e duplas por dentro |
| `AttributeError: 'NoneType' object has no attribute 'split'` | O `get()` devolveu `None` e o código tentou continuar | O XPath anterior não encontrou nada: teste-o sozinho, antes de acrescentar o `split()` |
| A célula de arquivos mostra `0` | O catálogo do Módulo 03 não foi baixado | Execute a célula que baixa o catálogo, na seção 4.5 |

---

# Retomada do Módulo 04

<img src="imagens/retomada.png" alt="Mapa de revisão do módulo: do texto à árvore, caminhos e condições, o que extrair, eixos, do navegador ao código e contêineres" width="1100">

### O que você aprendeu

- `Selector(text=html)` transforma o texto do HTML em uma árvore. O `xpath()` devolve **sempre uma lista**; `get()` tira o primeiro resultado e `getall()`, todos.
- No caminho, `/` desce um nível e `//` desce quantos forem necessários. Os **predicados** filtram: `[@class="x"]`, `[1]`, `[last()]`, `[text()="..."]`, `[contains(..., "...")]`.
- O final do XPath escolhe o que volta: o elemento, o texto (`text()`), um atributo (`@href`) ou o texto limpo de tudo o que está dentro (`normalize-space()`).
- Os **eixos** navegam a partir de um elemento: `..` (pai), `ancestor::`, `preceding-sibling::` e `following-sibling::`. O par **rótulo e valor** é o uso mais comum dos irmãos.
- O caminho até um bom XPath: selecionar o dado com o seletor de elementos, entender a árvore, apoiar-se em classes, atributos e textos, e testar no navegador, na busca do Inspetor ou com `$x()` no console. O XPath copiado do DevTools é um ponto de partida, não o resultado final.
- **O número de resultados é o teste**: 1 para um dado, um por item para uma lista. Resultados demais indicam que o XPath parou em um galho anterior.
- Para listas de itens, encontre os **contêineres** e busque dentro de cada um com XPaths relativos, começando com ponto.
- Os **seletores CSS** são um atalho para os casos simples; o XPath vai além deles.

### Teste seus conhecimentos

Clique em cada pergunta para ver a resposta.

<details>
<summary><b>1.</b> Qual é a diferença entre <code>//p/text()</code> e <code>//p</code>?</summary>

`//p` devolve os elementos inteiros, com as tags; `//p/text()` devolve só o texto que está diretamente dentro deles.
</details>

<details>
<summary><b>2.</b> Um XPath encontrou 20 elementos. O que devolvem o <code>get()</code> e o <code>getall()</code>?</summary>

O `get()` devolve o primeiro, como texto. O `getall()` devolve os 20, em uma lista de textos.
</details>

<details>
<summary><b>3.</b> Como pegar o valor ao lado do rótulo <code>Tax</code> em uma tabela de duas colunas?</summary>

Encontre o rótulo pelo texto e vá para o irmão seguinte: `//th[text()="Tax"]/following-sibling::td/text()`.
</details>

<details>
<summary><b>4.</b> Você quer o título de um livro, e o seu XPath encontra 12 elementos na busca do Inspetor. O que isso indica?</summary>

Que o XPath parou em um galho anterior, que se divide em vários. Desça mais na árvore, ou acrescente uma condição (uma classe, um rótulo), até a contagem ficar em 1.
</details>

<details>
<summary><b>5.</b> Dentro de um laço pelos contêineres, todos os livros devolvem o mesmo título. Qual é o erro?</summary>

O XPath de dentro do laço começa com `//`, sem o ponto, e por isso procura na página inteira. O certo é `.//`, que procura dentro do contêiner atual.
</details>

<details>
<summary><b>6.</b> Por que <code>//div/div[2]/ol/li[1]/article/div[2]/p[1]</code> é um XPath frágil?</summary>

Porque depende das **posições** de cada elemento no caminho. Se a página ganhar ou perder um elemento em qualquer nível, ele aponta para o lugar errado ou para nada. Classes e atributos que descrevem o dado são pontos de apoio mais estáveis.
</details>

### Glossário

| Termo | Significado |
|---|---|
| **parsel** | Biblioteca que transforma o texto do HTML em uma árvore e permite buscar nela |
| **Selector** | O objeto do parsel que representa a árvore, ou um elemento dela |
| **XPath** | Linguagem para descrever caminhos pela árvore de um documento |
| **Predicado** | Uma condição entre colchetes que filtra os elementos de um passo do XPath |
| **Eixo** | Uma direção de navegação a partir de um elemento: pai, irmãos, ancestrais, descendentes |
| **Irmão** (*sibling*) | Um elemento com o mesmo pai |
| **Contêiner** | O elemento que guarda todos os dados de um item, como o bloco de um livro |
| **XPath relativo** | Um XPath que começa com ponto e busca a partir de um elemento, e não da página inteira |
| **Seletor CSS** | Outra linguagem de busca, usada nos estilos das páginas; mais curta, porém mais limitada |
| **`$x()`** | Função do console do navegador que testa um XPath na página aberta |

---

## Próximo módulo

O **Módulo 05: Paginação e navegação** usa o XPath para **navegar**: encontrar o link da próxima página, completar links relativos como `a-light-in-the-attic_1000/index.html` e visitar a página de cada livro para coletar o que não aparece na listagem, como a categoria e a tabela de informações.

## Referências

Materiais consultados na elaboração deste módulo, agrupados pela parte em que foram usados.

**parsel** (em inglês)

- [Usage](https://parsel.readthedocs.io/en/latest/usage.html): criação de seletores, `get()` e `getall()`, XPaths relativos e seletores CSS com `::text` e `::attr()` (Partes 1, 5 e 6).
- [Selectors](https://docs.scrapy.org/en/latest/topics/selectors.html), na documentação do Scrapy, que usa o parsel por dentro (leitura complementar para o Módulo 10).

**XPath** (em inglês)

- MDN Web Docs: [XPath](https://developer.mozilla.org/en-US/docs/Web/XML/XPath), [Axes](https://developer.mozilla.org/en-US/docs/Web/XML/XPath/Reference/Axes) e [XPath functions](https://developer.mozilla.org/en-US/docs/Web/XML/XPath/Reference/Functions) (Partes 2 e 3).
- W3C. [XML Path Language (XPath) Version 1.0](https://www.w3.org/TR/xpath-10/), 1999. A especificação da versão usada pelo parsel (Partes 2 e 3).

**DevTools** (Parte 4)

- Mozilla: [Web Console Helpers](https://firefox-source-docs.mozilla.org/devtools-user/web_console/helpers/), com a função `$x()`, e [Examine and edit HTML](https://firefox-source-docs.mozilla.org/devtools-user/page_inspector/how_to/examine_and_edit_html/), com a opção de copiar o XPath (em inglês).
- Google: [Referência da API Console Utilitários](https://developer.chrome.com/docs/devtools/console/utilities?hl=pt-br), que vale também para o Edge.

**CSS** (Parte 6)

- MDN Web Docs: [Seletores CSS](https://developer.mozilla.org/pt-BR/docs/Web/CSS/Guides/Selectors).

**Sites usados nos exemplos:** a Livraria Exemplo (Módulo 02), [books.toscrape.com](https://books.toscrape.com), [quotes.toscrape.com](https://quotes.toscrape.com) e [scrapethissite.com](https://www.scrapethissite.com/pages/forms/).